In [ ]:
import numpy as np
from numba import cuda

In [ ]:
!uv pip install -q --system numba-cuda==0.4.0

In [ ]:
from numba import config
config.CUDA_ENABLE_PYNVJITLINK = 1
config.CUDA_LOW_OCCUPANCY_WARNINGS = 0

## Matrix multiplication

$$
\begin{bmatrix}
 a_{11} &  \dots  & a_{1n} \\
 \vdots &  \ddots & \vdots \\
 a_{m1} &  \dots  & a_{mn}
\end{bmatrix}
\times
\begin{bmatrix}
 b_{11} & \dots  & b_{1p} \\
 \vdots & \ddots & \vdots \\
 b_{n1} & \dots  & b_{np}
\end{bmatrix}
=\begin{bmatrix}
  c_{11} &  \dots  & c_{1m} \\
 \vdots &  \ddots & \vdots \\
 a_{n1} &  \dots  & c_{mp}
\end{bmatrix}
$$

$$
c_{ij}=\sum_{k=1}^na_{ik}b_{kj}
$$

### The algorithm in python

In [ ]:
def mult_mat(A, B):
    adim1, adim2 = A.shape
    bdim1, bdim2 = B.shape
    C=np.zeros((adim1,bdim2))
    #adim2 == bdim1
    for i in range(adim1):
        for j in range(bdim2):
            for k in range(adim2):
                C[i, j] += A[i, k] * B[k, j]
    return C

## Complete the kernel (internal loop)

In [ ]:
@cuda.jit
def k_mult_mat(A,B,C):
  i = cuda.threadIdx.x
  j = cuda.threadIdx.y
  n_rows_A, n_cols_A = A.shape
  n_rows_B, n_cols_B = B.shape

  if i < n_rows_A and j < n_cols_B:
      tmp = 0.0
      for k in range(n_cols_A):
          tmp += A[i, k] * B[k, j]
      C[i, j] = tmp

## Matrices initialisation on the host

In [ ]:
h_a=np.random.randn(32,32)
h_b=np.random.randn(32,32)
h_a=h_a.astype(np.float32)
h_b=h_b.astype(np.float32)
h_c=mult_mat(h_a, h_b)

## Complete memory allocation and copy to the device

In [ ]:
g_a=cuda.to_device(h_a)
g_b=cuda.to_device(h_b)
nrow, _ = h_a.shape
_, ncol = h_b.shape
g_c = cuda.device_array((nrow,ncol))

## Call the kernel with a well-sized block

In [ ]:
threadsperblock = (16, 16)
blockspergrid_x = int(np.ceil(nrow / threadsperblock[0]))
blockspergrid_y = int(np.ceil(ncol / threadsperblock[1]))
blockspergrid = (blockspergrid_x, blockspergrid_y)

k_mult_mat[blockspergrid, threadsperblock](g_a, g_b, g_c)

## Copy the results on the host

In [ ]:
h_c = g_c.copy_to_host()

## Check results

In [ ]:
print(np.dot(h_a,h_b))
print(h_c)
print(np.allclose(h_c, np.dot(h_a, h_b), atol=1e-5))